In [1]:
import tensorflow as tf

mnist = tf.keras.datasets.mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

In [2]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Dropout, Lambda, MaxPooling2D, BatchNormalization, ReLU, Reshape, GlobalAveragePooling2D
from tensorflow.keras import backend as K
import numpy as np
import random

# 1. NORMALIZE (Crucial! The model cannot learn if inputs are 0-255)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 2. SUBSET DATA (Satisfying "Small Portion" Rule)
# We take only 3,000 images (out of 60,000)
subset_size = 5000
indices = np.random.choice(len(X_train), subset_size, replace=False)
X_small = X_train[indices]
y_small = y_train[indices]

print(f"Data subset created. Using {len(X_small)} images.")

Data subset created. Using 5000 images.


In [3]:
def make_balanced_pairs(x, y):
    """Generates 50/50 Positive/Negative pairs from the small subset."""
    pairs = []
    labels = []

    num_classes = 10
    # Find where each digit is located in our small subset
    digit_indices = [np.where(y == i)[0] for i in range(num_classes)]

    # We iterate through each class to ensure we get samples of every number
    for d in range(num_classes):
        # Available images for this digit
        current_digit_indices = digit_indices[d]
        n = len(current_digit_indices) - 1

        if n < 1: continue # Skip if we don't have enough examples of this digit

        for i in range(n):
            # 1. Positive Pair (Same Digit)
            z1, z2 = current_digit_indices[i], current_digit_indices[i + 1]
            pairs += [[x[z1], x[z2]]]

            # 2. Negative Pair (Different Digit)
            # Pick a random different class
            inc = random.randrange(1, num_classes)
            dn = (d + inc) % num_classes

            # Make sure the other class has images
            if len(digit_indices[dn]) > 0:
                z3 = current_digit_indices[i]
                z4 = random.choice(digit_indices[dn]) # Random image from different class

                pairs += [[x[z3], x[z4]]]

                # Add labels (1 for same, 0 for different)
                labels += [1, 0]

    return np.array(pairs), np.array(labels).astype('float32')

# Generate the pairs
X_pairs, y_pairs = make_balanced_pairs(X_small, y_small)
print(f"Generated {len(y_pairs)} pairs for training.")

Generated 9980 pairs for training.


In [5]:
# --- 1. DEFINE DISTANCE LAYER ---
def euclidean_distance(vects):
    x, y = vects
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

# --- 2. DEFINE MODEL ARCHITECTURE ---
def create_base_network():
    # A simple, effective CNN for small MNIST data
    input = Input(shape=(28, 28, 1))
    x = Conv2D(32, (3, 3), activation='relu')(input)
    x = MaxPooling2D()(x)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x) # Feature vector
    return Model(input, x)

# Create the shared network
base_network = create_base_network()

# Define Inputs
img_A_inp = Input(shape=(28, 28), name='img_A_inp')
img_B_inp = Input(shape=(28, 28), name='img_B_inp')

# Reshape inputs to (28, 28, 1) because CNN expects 3 dimensions
img_A_reshaped = Reshape((28, 28, 1))(img_A_inp)
img_B_reshaped = Reshape((28, 28, 1))(img_B_inp)

# Pass both images through the SAME network
feature_vector_A = base_network(img_A_reshaped)
feature_vector_B = base_network(img_B_reshaped)

# Calculate Distance
distance = Lambda(euclidean_distance)([feature_vector_A, feature_vector_B])

# Final Model
model = Model(inputs=[img_A_inp, img_B_inp], outputs=distance)

# --- 3. LOSS & METRICS ---
def contrastive_loss(y_true, y_pred):
    margin = 1.0
    sq_pred = K.square(y_pred)
    margin_sq = K.square(K.maximum(margin - y_pred, 0))
    return K.mean(y_true * sq_pred + (1 - y_true) * margin_sq)

# ✅ FIX: Squeeze y_pred from (batch,1) to (batch,) before comparing
# Previously, keepdims=True in euclidean_distance made y_pred shape (batch,1)
# while y_true was (batch,), causing a broadcasting bug that kept accuracy stuck at 50%
def contrastive_accuracy(y_true, y_pred):
    y_pred_flat = K.squeeze(y_pred, axis=-1)  # (batch,1) -> (batch,)
    return K.mean(tf.equal(y_true, tf.cast(y_pred_flat < 0.5, y_true.dtype)))

# --- 4. TRAIN ---
model.compile(loss=contrastive_loss, optimizer='adam', metrics=[contrastive_accuracy])

print("Starting Training...")
# Training for 20 epochs because data is small (needs more passes to learn)
history = model.fit(
    [X_pairs[:, 0], X_pairs[:, 1]],
    y_pairs,
    batch_size=64,
    epochs=40,
    validation_split=0.2
)

# --- 5. SAVE ---
model.save('siamese_model_contrastive.h5')
print("Model saved successfully!")

Starting Training...
Epoch 1/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 134ms/step - contrastive_accuracy: 0.8469 - loss: 0.1296 - val_contrastive_accuracy: 0.8029 - val_loss: 0.1445
Epoch 2/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 117ms/step - contrastive_accuracy: 0.9741 - loss: 0.0425 - val_contrastive_accuracy: 0.7516 - val_loss: 0.1621
Epoch 3/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 121ms/step - contrastive_accuracy: 0.9908 - loss: 0.0252 - val_contrastive_accuracy: 0.6965 - val_loss: 0.2148
Epoch 4/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 119ms/step - contrastive_accuracy: 0.9951 - loss: 0.0161 - val_contrastive_accuracy: 0.6699 - val_loss: 0.2596
Epoch 5/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 22s 131ms/step - contrastive_accuracy: 0.9965 - loss: 0.0109 - val_contrastive_accuracy: 0.6621 - val_loss: 0.2713
Epoch 6/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 19s 120ms/step - contrastive_accuracy: 0.9984 - loss: 0.0086 - val_contrastive_accuracy: 0.6530 - val_loss: 0.2970
Epoch 7/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 21s 121ms

Model saved successfully!
